Trigram langauge model

In [7]:
words = open('names.txt','r').read().splitlines()

In [8]:
len(words)

32033

In [9]:
words[:4]

['emma', 'olivia', 'ava', 'isabella']

In [10]:
import torch

In [11]:
N = torch.zeros((27,27,27), dtype = torch.int32)
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [12]:
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs,chs[1:],chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    N[ix1,ix2,ix3] += 1


p = (N+1).float()
p = p/p.sum(2,keepdims=True)


In [13]:
p[1,1].sum()

tensor(1.0000)

In [14]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
  out = ['.','.']
  ix = 0
  while True:
    p1 = p[stoi[out[-2]],stoi[out[-1]]]
    ix = torch.multinomial(p1,num_samples=1,replacement=True,generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out[2:]))


ce.
za.
zogh.
uriana.
kaydnevonimittain.


In [15]:
log_likelihood = 0.0
n = 0

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs,chs[1:],chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    prob =p[ix1,ix2,ix3]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1

print(f'{log_likelihood=}')
null = -log_likelihood
print(f'{null=}')
print(f'{null/n}')



log_likelihood=tensor(-410414.9688)
null=tensor(410414.9688)
2.092747449874878


In [33]:
xs ,ys = [],[]

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs,chs[1:],chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    xs.append((ix1,ix2))
    ys.append(ix3)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print(f'{num}')




392226


In [17]:
xs

tensor([[ 0,  5],
        [ 5, 13],
        [13, 13],
        ...,
        [26, 25],
        [25, 26],
        [26, 24]])

In [18]:
ys

tensor([13, 13,  1,  ..., 26, 24,  0])

In [40]:
g = torch.Generator().manual_seed(2147483647)
w = torch.randn((54,27) , generator=g, requires_grad=True)

In [46]:
import torch.nn.functional as F
for k in range(1000):
  xenc = F.one_hot(xs,num_classes=27).float()
  logits = xenc.view(-1,54) @ w
  counts = logits.exp()
  prob = counts / counts.sum(1,keepdims=True)
  loss = -prob[torch.arange(ys.shape[0]),ys].log().mean()

  w.grad = None
  loss.backward()

  w.data += -50 * w.grad


print(loss.item())


2.2379188537597656
